# Notebook 05a — Wealth signals: the executable build (reproduce the card state)

**ATLAS: Aligned Three-Layer Architecture for Semantics**  
FSI Semantic Layer Workshop on AWS — Workshop 2

---

This is the **executable companion** to `05_wealth_signals.ipynb` (which *teaches* the
derivation, read-only). This notebook *runs* it: it loads the Workshop 2 signal-type
concepts into the SLGD and derives the **No Advisor Coverage** signal, so the referral
card reproduces its two honest signals on your account.

**Run order matters.** This notebook must run **after** Workshop 1's signal derivation
(`agentic-semantic-layer/notebooks/05_entity_resolution.ipynb`, the cell that derives
Large Deposit Pattern + Household Aggregation into the SLGD). No Advisor Coverage uses
*gate C* — it fires only for a customer who **already** produces another wealth signal.
If you run this before WS1's signals are persisted, it fires for nobody.

**One source of truth.** The derivation logic is NOT written here — it is *imported*
from the committed scripts (`scripts/load-ws2-ontology-concepts.py`,
`scripts/derive-no-advisor-coverage.py`). This notebook orchestrates and teaches; the
scripts own the rule. That keeps the runnable path and the deployed behaviour identical.

In [ ]:
# Setup: dependencies + a signed SLGD transport. The transport (read/update) is the
# only thing defined here; ALL derivation logic is imported from scripts/ below.
import sys, subprocess
subprocess.check_call([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check',
                       'rdflib>=7.0.0','pyshacl>=0.25.0','boto3>=1.43.0'], cwd='/tmp')

import json, urllib.parse, urllib.request, os

# The SLGD is reached via the atlas-sparql-mcp runtime (JWT) or, in-VPC from Studio,
# the WS1 SigV4 helper. Set SPARQL_MCP_ARN + a bearer token (banker JWT) in your env,
# or replace _run_query/_run_update with the WS1 sparql_query/sparql_update_slgd helpers.
SPARQL_MCP_ARN = os.environ.get('SPARQL_MCP_ARN', '')
BEARER = os.environ.get('ATLAS_BEARER_TOKEN', '')
PERSONA = 'atlas-consumer-banker'

def _endpoint(arn):
    return f"https://bedrock-agentcore.us-east-1.amazonaws.com/runtimes/{urllib.parse.quote(arn, safe='')}/invocations"

def _invoke(payload):
    req = urllib.request.Request(_endpoint(SPARQL_MCP_ARN), data=json.dumps(payload).encode(), method='POST')
    req.add_header('Content-Type','application/json'); req.add_header('Authorization', f'Bearer {BEARER}')
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.loads(r.read())

def _run_query(sparql):
    out = _invoke({'operation':'query','sparql':sparql,'persona_claim':PERSONA,'graph_tier':'slgd'})
    return out.get('rows', [])

def _run_update(sparql):
    out = _invoke({'operation':'update','sparql':sparql,'persona_claim':PERSONA})
    if out.get('status') == 'error': raise RuntimeError(out.get('message','update failed'))
    return out

print('Transport ready. SPARQL_MCP_ARN set:', bool(SPARQL_MCP_ARN), '| token set:', bool(BEARER))

## Import the committed derivation logic (one source of truth)

The scripts live under `use-case-applications/scripts/` with hyphenated filenames, so
we load them by path with `importlib`. We import *functions* only — the scripts run no
side effects on import (all I/O is behind their `if __name__ == "__main__"` guards).

In [ ]:
import importlib.util
from pathlib import Path

SCRIPTS = Path.cwd().parents[1] / 'scripts'   # use-case-applications/scripts

def _load(modname, filename):
    spec = importlib.util.spec_from_file_location(modname, SCRIPTS / filename)
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
    return mod

loader = _load('load_ws2_concepts', 'load-ws2-ontology-concepts.py')
nac    = _load('derive_nac', 'derive-no-advisor-coverage.py')
print('Imported:', 'build_insert' in dir(loader), '|',
      all(hasattr(nac, f) for f in ('derive','validate_signals','insert_query')))

## Precondition — Workshop 1 signals must already be persisted

Gate C reads `?customer atlas:producesSignal ?anySig`. If WS1's Large Deposit Pattern /
Household Aggregation signals are not yet in the SLGD, No Advisor Coverage fires for
nobody. This cell **stops** the notebook if they are absent.

In [ ]:
# Precondition: WS1 LDP/HH signals exist (gate C depends on them).
rows = _run_query('''PREFIX atlas: <https://github.com/your-org/atlas/ontology#>
SELECT (COUNT(DISTINCT ?s) AS ?n) WHERE {
  ?c atlas:producesSignal ?s . ?s atlas:hasSignalType ?t
  FILTER(?t IN (atlas:LargeDepositPattern, atlas:HouseholdAggregationSignal)) }''')
ws1_signals = int(rows[0]['n']) if rows else 0
assert ws1_signals > 0, (
    'No WS1 LargeDepositPattern/HouseholdAggregation signals in the SLGD. Run Workshop 1 '
    'nb05 signal derivation FIRST — gate-C NoAdvisorCoverage cannot fire without them.')
print(f'Precondition OK: {ws1_signals} WS1 signals present — gate C can match.')

## Teardown (run these to RESET, e.g. to prove reproduction or to revert)

Both are surgical and proven safe (Pass 2b): the concept DELETE matches only the
`atlas-part-2:` namespace (no Workshop 1 triple is under it); the signal DELETE matches
only `NoAdvisorCoverageSignal` carrying the `signal-derivation-run` provenance stamp
(never promoted data, never WS1's LargeDepositPattern). Run them to return to a clean
state, then re-run the load + derive cells to reproduce from scratch.

In [ ]:
# TEARDOWN 1 — remove the WS2 atlas-part-2: concept triples (namespace-scoped; 0 WS1 match).
TEARDOWN_CONCEPTS = '''
PREFIX atlas: <https://github.com/your-org/atlas/ontology#>
PREFIX prov: <http://www.w3.org/ns/prov#>
DELETE { ?s ?p ?o }
WHERE { ?s ?p ?o . FILTER(STRSTARTS(STR(?s), "https://github.com/your-org/atlas/ontology/part2#")) }'''

# TEARDOWN 2 — remove the derived NoAdvisorCoverage signals (type + derivation-run stamp).
TEARDOWN_NAC = '''
PREFIX atlas: <https://github.com/your-org/atlas/ontology#>
PREFIX atlas-part-2: <https://github.com/your-org/atlas/ontology/part2#>
PREFIX prov: <http://www.w3.org/ns/prov#>
DELETE { ?signal ?sp ?so . ?customer atlas:producesSignal ?signal . }
WHERE { ?signal atlas:hasSignalType atlas-part-2:NoAdvisorCoverageSignal ;
                prov:wasGeneratedBy <https://github.com/your-org/atlas/instance#signal-derivation-run> ;
                ?sp ?so .
        OPTIONAL { ?customer atlas:producesSignal ?signal } }'''

# Uncomment to reset before reproducing:
# _run_update(TEARDOWN_NAC); _run_update(TEARDOWN_CONCEPTS); print('Reset done.')
print('Teardown queries defined. Uncomment the line above to reset.')

## Step 1 — load the Workshop 2 signal-type concepts

`load-ws2-ontology-concepts.py` parses `ontology-extensions/signal-types.ttl` into one
`INSERT DATA`. Idempotent: the concepts have fixed URIs, so re-running re-inserts the
same triples (RDF set semantics) and the graph is unchanged. Without this, the No
Advisor Coverage signal's type — and the card's label — would not resolve.

In [ ]:
# Converge the SLGD's atlas-part-2: concepts to the file: build_reload() returns
# [DELETE the part2# namespace, INSERT the file's triples]. Running both in order makes
# the load idempotent even after a concept's literal (e.g. a comment) was edited — a
# bare INSERT would leave the old + new value both. Requires DeleteDataViaQuery (granted).
for stmt in loader.build_reload():
    _run_update(stmt)

rows = _run_query('SELECT (COUNT(*) AS ?n) WHERE { ?s ?p ?o '
                  'FILTER(STRSTARTS(STR(?s), "https://github.com/your-org/atlas/ontology/part2#")) }')
print('atlas-part-2: concept triples loaded:', rows[0]['n'], '(expect 25)')
rows = _run_query('PREFIX atlas-part-2: <https://github.com/your-org/atlas/ontology/part2#> '
                  'PREFIX skos: <http://www.w3.org/2004/02/skos/core#> '
                  'SELECT ?l WHERE { atlas-part-2:NoAdvisorCoverageSignal skos:prefLabel ?l }')
print('NoAdvisorCoverageSignal label resolves to:', rows[0]['l'] if rows else '(MISSING)')

## Step 2 — derive No Advisor Coverage (validate-before-write)

`derive()` runs gate C against the live SLGD (uncovered **and** already-signalled),
`validate_signals()` is the pyshacl gate against Workshop 1's `atlas-shapes.ttl`
(returns the triples only if they conform, raises otherwise), and `insert_query()`
wraps them for the write. We INSERT only what validated — the same derive-don't-insert
discipline taught in `05_wealth_signals.ipynb`. Expect **40** signals; **0 would mean
the ordering hazard** (WS1 signals not persisted) — the cell stops in that case.

In [ ]:
# derive() needs a query callable; hand it our signed _run_query.
validated_nt = nac.derive(_run_query)        # gate-C CONSTRUCT + pyshacl gate (raises if non-conformant)
n_signals = len(validated_nt) // 4           # 4 triples per signal
assert n_signals > 0, ('Gate C produced 0 signals — WS1 LDP/HH likely not persisted before this ran '
                       '(the ordering hazard). Run WS1 signal derivation first, then re-run.')

_run_update(nac.insert_query(validated_nt))  # write only the SHACL-validated triples

rows = _run_query('PREFIX atlas: <https://github.com/your-org/atlas/ontology#> '
                  'PREFIX atlas-part-2: <https://github.com/your-org/atlas/ontology/part2#> '
                  'SELECT (COUNT(DISTINCT ?s) AS ?n) WHERE { ?s atlas:hasSignalType atlas-part-2:NoAdvisorCoverageSignal }')
print(f'NoAdvisorCoverage signals derived + validated + written: {rows[0]["n"]} (expect 40)')

## Verification — the demo customer now has both signals

c6b6e4ad (the coverage-gap persona) should now produce **both** Large Deposit Pattern
(WS1) and No Advisor Coverage (this notebook). That is the two-signal card the
wholesale referral screen renders.

In [ ]:
C = 'https://github.com/your-org/atlas/instance#customer-c6b6e4ad-e7ea-8d5a-31a3-ac4d828c37e7-resolved'
rows = _run_query(f'''PREFIX atlas: <https://github.com/your-org/atlas/ontology#>
SELECT ?type WHERE {{ <{C}> atlas:producesSignal ?s . ?s atlas:hasSignalType ?type }}''')
types = sorted(r['type'].split('#')[-1].split('/')[-1] for r in rows)
print('c6b6e4ad produces:', types)
assert any('LargeDepositPattern' in t for t in types), 'LDP missing — WS1 derivation not run?'
assert any('NoAdvisorCoverage' in t for t in types), 'NAC missing — Step 2 did not fire for c6b6e4ad'
print('OK — the two-signal card reproduces for the demo customer.')

## What just changed

Your SLGD now contains the Workshop 2 signal-type concepts (25 triples) and the derived
No Advisor Coverage signals (40, gate-C scoped), reproduced **from the committed
scripts** — not hand-run, not inlined. The referral card's two honest signals are now
rebuildable by running this notebook after Workshop 1's signal derivation, on any
account. The read-only `05_wealth_signals.ipynb` explains *why* each signal is derived
(and why a third is refused); this notebook *produces* them. To reset, run the teardown
cells; to reproduce, re-run Steps 1–2 (both idempotent / re-derivable).